# Mem0 集成模式

> **给每个用户一份持久、自我改进的记忆。抽取流程、向量库、去重逻辑，都不必自己再建。**

想象一位私人助理，每次开会都做笔记。每场谈话结束后，他会记下关键事实：你的偏好、你的日程、你的人脉。下一次会面前，他会先翻一遍笔记，以便更好地帮到你。Mem0 就是你的 AI 智能体的那位助理，而且完全自动化。

大多数智能体记忆方案要求你手工接好三套系统。第一，基于 LLM 的抽取步骤，从对话中提取事实。第二，向量数据库（一种按语义相似度查找条目的数据库）负责存储。第三，处理矛盾和重复的核对合并逻辑。**Mem0** 把这一切收进一个 `add()` / `search()` / `get_all()` API。

Mem0 的开源库在底层做三件事：
1. 把对话消息交给 LLM，**抽取关键事实、偏好和指令**。
2. 对照已有记忆**去重并解决矛盾**。
3. 把结果存进向量数据库（默认 Qdrant），支持**语义检索**（按含义查找记忆，而不是匹配精确关键词）。

最终你得到一个**按用户隔离的记忆层**，而且越用越聪明。新信息到来时，记忆会被自动合并、更新和打磨。

**学完本 notebook，你将能够：**
- 安装并配置 Mem0，用 OpenAI 作为 LLM 后端。
- 从对话消息和纯文本中添加记忆。
- 搜索、检索、更新和删除记忆。
- 构建一个借助 Mem0 个性化回复的记忆增强智能体循环。
- 判断什么时候该用 Mem0，什么时候该自己搭记忆系统。


## 核心概念

- **`Memory.add()`：** 接受对话消息（或纯文本）。运行基于 LLM 的抽取，识别值得记住的事实。对照已有记忆检查重复和矛盾，然后存储结果。这一次调用就替代了自定义的抽取 + 嵌入 + upsert 流程。
- **`Memory.search()`：** 用自然语言查询对已存记忆做语义搜索。返回带相关性分数的排序结果。支持按 `user_id`、`agent_id` 和元数据过滤。
- **`Memory.get_all()`：** 检索某个用户、智能体或会话范围内的全部记忆。适合展示用户的完整记忆画像，或用来调试。
- **`Memory.update()` / `Memory.delete()`：** 按 ID 显式修改或删除单条记忆。当你确定某个事实已经变化时（比如用户搬了城市），更新路径很有用。
- **按用户隔离：** 每个操作都以 `user_id` 划定范围。一个用户的记忆绝不会泄入另一个用户的上下文。可选的 `agent_id` 和 `run_id` 提供更细粒度的隔离。
- **自动冲突解决：** 当你 `add()` 一条与已有记忆矛盾的记忆时（比如「我从纽约搬到了伦敦」），Mem0 的 LLM 流程会检测到冲突，直接更新已存记忆，而不是制造重复。
- **自我改进的记忆：** 随着一次次 `add()`，Mem0 会合并并打磨相关记忆。无需人工整理，记忆库会随时间变得更准确、更精炼。


## 架构

<p align="center">
  <img src="../../images/diagrams/25_mem0_patterns.svg" alt="架构图" width="720"/>
</p>

<details><summary>Mermaid 源码</summary>

```mermaid
flowchart LR
    subgraph App["Agent / Application"]
        A["User message"] --> B["Agent generates\nresponse"]
        B --> C["Response to user"]
    end

    subgraph Mem0Layer["Mem0 Memory Layer"]
        D["memory.add()\n─────────\nLLM extraction\nConflict resolution\nDeduplication"]
        E["memory.search()\n─────────\nSemantic search\nRanked results"]
        F[("Vector Store\n(Qdrant)\n─────────\nEmbeddings\nMetadata\nUser scopes")]
    end

    subgraph Ops["Management"]
        G["memory.get_all()\nmemory.update()\nmemory.delete()"]
    end

    A -->|"conversation turns"| D
    D -->|"extracted facts"| F
    A -->|"query"| E
    E -->|"relevant memories"| B
    F <-->|"store / retrieve"| E
    G <--> F

    style F fill:#4f46e5,color:#fff
    style D fill:#059669,color:#fff
    style E fill:#d97706,color:#fff
    style B fill:#6366f1,color:#fff
```

</details>

**每轮对话的数据流：**
1. 用户发来消息。智能体调用 `memory.search()` 检索相关上下文。
2. 智能体以检索到的记忆为上下文生成回复。
3. 完整的一轮对话传给 `memory.add()`，抽取并存储新事实。
4. Mem0 的 LLM 流程抽取事实、解决冲突，并更新向量库。

## 环境准备

安装所需的包。默认情况下，Mem0 的抽取 LLM 和嵌入都用 OpenAI。


In [ ]:
# 安装所需包（只需运行一次）
%pip install -q mem0ai python-dotenv deepagents langchain-openai

导入 Mem0，并从 `.env` 文件加载你的 API key。
Mem0 会自动读取 `LLM_API_KEY`，用于 LLM 和嵌入调用。


In [ ]:
import os
import json

from dotenv import load_dotenv

load_dotenv()  # 从 .env 读取 LLM_API_KEY / LLM_BASE_URL / LLM_MODEL_NAME

from langchain_openai import ChatOpenAI
from deepagents import create_deep_agent

assert os.getenv("LLM_API_KEY"), "Set LLM_API_KEY in your .env file"

# 任何 OpenAI 兼容端点都可以：用官方 OpenAI 时省略 base_url 即可
llm = ChatOpenAI(
    model=os.getenv("LLM_MODEL_NAME", "gpt-4o-mini"),
    api_key=os.getenv("LLM_API_KEY"),
    base_url=os.getenv("LLM_BASE_URL"),
)



def to_text(content) -> str:
    """把字符串或内容块列表统一转成纯文本。"""
    if isinstance(content, str):
        return content
    return "".join(
        block.get("text", "") for block in content if isinstance(block, dict)
    )


# Graphiti / Mem0 等框架默认读取 OPENAI_* 环境变量；
# 这里把统一端点映射过去，让框架内部调用也走同一个网关。
os.environ.setdefault("OPENAI_API_KEY", os.getenv("LLM_API_KEY", ""))
os.environ.setdefault("OPENAI_BASE_URL", os.getenv("LLM_BASE_URL", ""))
if os.getenv("LLM_MODEL_NAME"):
    os.environ.setdefault("OPENAI_MODEL", os.getenv("LLM_MODEL_NAME"))

print("✓ Model ready")


## 核心实现

我们逐步走一遍每个 Mem0 操作：

1. **初始化：** 创建 `Memory` 实例（默认使用本地 Qdrant + OpenAI）。
2. **添加：** 从对话消息中存入记忆。
3. **搜索：** 用语义查询检索相关记忆。
4. **获取全部：** 列出某用户的完整记忆画像。
5. **更新：** 修改已有记忆。
6. **删除：** 移除指定记忆。
7. **历史：** 查看一条记忆的变更历史。


In [ ]:
# 用默认配置初始化 Mem0
# 默认配置：Qdrant（本地磁盘）+ OpenAI gpt-4o-mini + text-embedding-3-small
m = Memory()

print("\u2713 Memory instance created")
print("  Vector store: Qdrant (local)")
print("  LLM: OpenAI (default)")
print("  Embedder: OpenAI text-embedding-3-small")

### 从对话中添加记忆

把对话消息传给 `m.add()`。Mem0 的 LLM 流程会自动抽取关键事实。
每次调用都以 `user_id` 划定范围，所以不同用户的记忆互不混杂。


In [ ]:
# ── 从对话消息中添加记忆 ──
# Mem0 会自动抽取事实，你只需传入对话。

USER_ID = "alice"

# 对话 1：饮食偏好
messages_1 = [
    {"role": "user", "content": "I'm a vegetarian and I love Italian food. My favorite dish is mushroom risotto."},
    {"role": "assistant", "content": "Great taste! Mushroom risotto is a classic. I'll remember your preferences."},
]
result_1 = m.add(messages_1, user_id=USER_ID)
print("Add result (food preferences):")
print(json.dumps(result_1, indent=2))

print()

# 对话 2：工作背景
messages_2 = [
    {"role": "user", "content": "I'm a senior data scientist at Google. I mostly work with Python and PyTorch."},
    {"role": "assistant", "content": "Nice! Data science at Google, exciting work. I'll keep that in mind."},
]
result_2 = m.add(messages_2, user_id=USER_ID)
print("Add result (work context):")
print(json.dumps(result_2, indent=2))

print()

# 对话 3：个人信息
messages_3 = [
    {"role": "user", "content": "I live in San Francisco. I moved here from New York last year. I have a golden retriever named Max."},
    {"role": "assistant", "content": "SF is lovely! And golden retrievers are the best. How's Max adjusting to the new city?"},
]
result_3 = m.add(messages_3, user_id=USER_ID)
print("Add result (personal details):")
print(json.dumps(result_3, indent=2))

### 搜索记忆

用 `m.search()` 按含义查找记忆，而不是匹配精确关键词。
Mem0 会把你的查询变成嵌入，返回最相似的已存事实，并按相关性排序。


In [ ]:
# ── 按语义查询搜索记忆 ──
# Mem0 按相似度排序，返回最相关的记忆。

print("=== Search: 'What food does Alice like?' ===\n")
food_results = m.search("What food does Alice like?", user_id=USER_ID)
for mem in food_results.get("results", food_results if isinstance(food_results, list) else []):
    if isinstance(mem, dict):
        print(f"  \u2022 {mem.get('memory', mem.get('text', str(mem)))}")

print()

print("=== Search: 'What does Alice do for work?' ===\n")
work_results = m.search("What does Alice do for work?", user_id=USER_ID)
for mem in work_results.get("results", work_results if isinstance(work_results, list) else []):
    if isinstance(mem, dict):
        print(f"  \u2022 {mem.get('memory', mem.get('text', str(mem)))}")

print()

print("=== Search: 'pets' ===\n")
pet_results = m.search("pets", user_id=USER_ID)
for mem in pet_results.get("results", pet_results if isinstance(pet_results, list) else []):
    if isinstance(mem, dict):
        print(f"  \u2022 {mem.get('memory', mem.get('text', str(mem)))}")

### 列出全部记忆

调用 `m.get_all()` 查看某用户存储的每一条记忆。
这对调试很有用，也可以用来向用户展示完整画像。


In [ ]:
# ── 获取某用户的全部记忆 ──
# 适合查看完整的记忆画像。

all_memories = m.get_all(user_id=USER_ID)

# 兼容列表和字典两种响应格式
memory_list = all_memories.get("results", all_memories) if isinstance(all_memories, dict) else all_memories

print(f"=== All memories for '{USER_ID}' ({len(memory_list)} total) ===\n")
for i, mem in enumerate(memory_list, 1):
    mem_id = mem.get("id", "N/A")
    mem_text = mem.get("memory", mem.get("text", str(mem)))
    print(f"  {i}. [{mem_id[:8]}...] {mem_text}")

# 保存一个记忆 ID，供后面的更新 / 删除演示使用
if memory_list:
    SAMPLE_MEMORY_ID = memory_list[0].get("id")
    print(f"\n\u2713 Saved memory ID for later: {SAMPLE_MEMORY_ID[:12]}...")

### 自动冲突解决

当你添加的事实与已有记忆矛盾时，Mem0 会检测到冲突。
它会更新已存的记忆，而不是制造重复。
看看 Alice 的所在地如何从旧金山变成伦敦。


In [ ]:
# ── 自动冲突解决 ──
# 当新事实与已有记忆矛盾时，Mem0 会更新旧记忆。

print("=== Before: Alice's location ===")
location_before = m.search("Where does Alice live?", user_id=USER_ID)
for mem in (location_before.get("results", location_before) if isinstance(location_before, dict) else location_before):
    if isinstance(mem, dict):
        print(f"  \u2022 {mem.get('memory', mem.get('text', str(mem)))}")

# Alice 搬到了伦敦
messages_move = [
    {"role": "user", "content": "I just moved to London! Left San Francisco last week."},
    {"role": "assistant", "content": "Exciting move! How are you finding London so far?"},
]
result_move = m.add(messages_move, user_id=USER_ID)
print(f"\nUpdate result:")
print(json.dumps(result_move, indent=2))

print("\n=== After: Alice's location ===")
location_after = m.search("Where does Alice live?", user_id=USER_ID)
for mem in (location_after.get("results", location_after) if isinstance(location_after, dict) else location_after):
    if isinstance(mem, dict):
        print(f"  \u2022 {mem.get('memory', mem.get('text', str(mem)))}")

print("\n\u2713 Mem0 resolved the contradiction: location updated, not duplicated.")

### 显式更新与删除

你也可以按 ID 修改或删除特定记忆。
当你确切知道哪个事实变了，用 `m.update()`。
当某条记忆应当彻底移除时，用 `m.delete()`。


In [ ]:
# ── 显式更新与删除 ──

# 更新：按 ID 修改指定记忆
if SAMPLE_MEMORY_ID:
    print(f"=== Updating memory {SAMPLE_MEMORY_ID[:12]}... ===")
    update_result = m.update(memory_id=SAMPLE_MEMORY_ID, data="Alice is a strict vegan who loves Italian cuisine")
    print(f"Update result: {update_result}")

    # 验证更新结果
    print("\nAfter update:")
    updated = m.search("food preferences", user_id=USER_ID)
    for mem in (updated.get("results", updated) if isinstance(updated, dict) else updated):
        if isinstance(mem, dict):
            print(f"  \u2022 {mem.get('memory', mem.get('text', str(mem)))}")

print()

# 删除：移除指定记忆
# 先添加一条临时记忆，用来演示删除
temp = m.add("Alice temporarily likes sushi", user_id=USER_ID)
temp_list = temp.get("results", temp) if isinstance(temp, dict) else temp
if isinstance(temp_list, list) and temp_list:
    temp_id = temp_list[0].get("id", temp_list[0].get("memory_id"))
elif isinstance(temp_list, dict):
    temp_id = temp_list.get("id", temp_list.get("memory_id"))
else:
    temp_id = None

if temp_id:
    print(f"=== Deleting temporary memory {str(temp_id)[:12]}... ===")
    delete_result = m.delete(memory_id=temp_id)
    print(f"Delete result: {delete_result}")
    print("\u2713 Memory deleted")

## 记忆增强的智能体循环

现在我们来构建一个实用的智能体，用 Mem0 个性化它的回复。这个模式分三步：

1. **检索：** 回复之前，先在 Mem0 中搜索与用户消息相关的记忆。
2. **生成：** 把检索到的记忆放进 system 提示词，让 LLM 据此个性化回复。
3. **存储：** 对话结束后，把这段对话传给 `memory.add()`，捕获新事实。

这样就形成了**自我强化的循环**。每次对话都在充实记忆，每次回复都能受益于积累的知识。


In [ ]:


class Mem0Agent:
    """一个使用 Mem0 实现持久化、个性化记忆的对话智能体。"""

    def __init__(self, memory: Memory, model=llm, user_id: str = "default"):
        self.memory = memory
        self.model = model
        self.user_id = user_id
        self.conversation: list[dict] = []

`chat` 方法把三步模式串起来：检索、生成、存储。
回答之前，先搜索 Mem0 获取相关上下文。
回答之后，把这段对话交回 Mem0，捕获新事实。


In [ ]:
def _chat(self, user_message: str) -> str:
    """处理用户消息：检索记忆、生成回复、存储新记忆。"""

    # 第 1 步：检索相关记忆
    relevant_memories = self.memory.search(user_message, user_id=self.user_id)
    mem_list = relevant_memories.get("results", relevant_memories) if isinstance(relevant_memories, dict) else relevant_memories

    # 把记忆格式化后放进 system 提示词
    if mem_list:
        memory_context = "\n".join(
            f"- {mem.get('memory', mem.get('text', str(mem)))}"
            for mem in mem_list
            if isinstance(mem, dict)
        )
        memory_block = (
            f"\n\nYou have these memories about the user (use them to personalize your response):\n"
            f"{memory_context}"
        )
    else:
        memory_block = ""

    system_prompt = (
        "You are a helpful, friendly assistant with a great memory. "
        "You remember details about the user from past conversations and use them "
        "to provide personalized, relevant responses. Reference specific things you "
        "remember when appropriate. It shows you care."
        f"{memory_block}"
    )

    # 第 2 步：构建消息并生成回复
    self.conversation.append({"role": "user", "content": user_message})

    # 记忆上下文每轮变化，系统提示词每轮都不同，所以每轮重建智能体
    agent = create_deep_agent(model=self.model, system_prompt=system_prompt)
    result = agent.invoke({"messages": list(self.conversation[-10:])})
    assistant_message = to_text(result["messages"][-1].content)
    self.conversation.append({"role": "assistant", "content": assistant_message})

    # 第 3 步：把这段对话存成新记忆
    self.memory.add(
        [
            {"role": "user", "content": user_message},
            {"role": "assistant", "content": assistant_message},
        ],
        user_id=self.user_id,
    )

    return assistant_message

Mem0Agent.chat = _chat


`show_memories` 辅助方法会打印当前用户的全部已存记忆。
每次演示之后我们都会调用它，看看 Mem0 抽取出了什么。


In [ ]:
def _show_memories(self) -> None:
    """展示该用户的所有已存记忆。"""
    all_mem = self.memory.get_all(user_id=self.user_id)
    mem_list = all_mem.get("results", all_mem) if isinstance(all_mem, dict) else all_mem
    print(f"\n=== Memories for '{self.user_id}' ({len(mem_list)} total) ===")
    for i, mem in enumerate(mem_list, 1):
        print(f"  {i}. {mem.get('memory', mem.get('text', str(mem)))}")


print("\u2713 Mem0Agent class defined")

Mem0Agent.show_memories = _show_memories


### 演示：多轮个性化对话

我们创建一个新用户（Bob），进行三轮对话。
智能体应该记住 Bob 的信息，并据此个性化回复。
对话结束后，我们检查 Mem0 存了什么。


In [ ]:
# ── 演示：多轮个性化对话 ──

# 这个演示用另一个 user_id，从零开始
agent = Mem0Agent(memory=m, user_id="bob")

# 第 1 轮：Bob 自我介绍
print("=" * 60)
print("Turn 1")
print("=" * 60)
user_msg = "Hi! I'm Bob. I'm a software engineer and I love hiking on weekends."
print(f"User: {user_msg}\n")
response = agent.chat(user_msg)
print(f"Agent: {response}")

print()

# 第 2 轮：更多偏好
print("=" * 60)
print("Turn 2")
print("=" * 60)
user_msg = "I've been learning Rust lately. Also, I'm training for a half marathon in October."
print(f"User: {user_msg}\n")
response = agent.chat(user_msg)
print(f"Agent: {response}")

print()

# 第 3 轮：智能体应能回忆起前面的上下文
print("=" * 60)
print("Turn 3")
print("=" * 60)
user_msg = "Can you suggest some weekend activities for me?"
print(f"User: {user_msg}\n")
response = agent.chat(user_msg)
print(f"Agent: {response}")

# 查看 Mem0 存了什么
agent.show_memories()

### 模拟新会话

现在我们新建一个全新的智能体实例，但沿用同一个 `user_id`。
这模拟了 Bob 几天后回来的场景。
记忆持久保存在 Mem0 里，所以智能体应该记得之前的一切。


In [ ]:
# ── 模拟新会话 ──
# 新建一个智能体实例，user_id 不变。
# 记忆从上一段对话延续下来。

print("=" * 60)
print("NEW SESSION (fresh agent, same user_id)")
print("=" * 60)

agent2 = Mem0Agent(memory=m, user_id="bob")

# Bob 几天后回来，智能体应该还记得他
user_msg = "Hey, I just finished my first Rust project! What do you remember about me?"
print(f"\nUser: {user_msg}\n")
response = agent2.chat(user_msg)
print(f"Agent: {response}")

print("\n\u2713 Memories persisted across sessions. The agent remembers Bob!")

## 自定义配置

Mem0 的默认配置（OpenAI + 本地 Qdrant）适合开发。要上生产环境或换一家 LLM 供应商时，每个组件都可以自定义：

```python
from mem0 import Memory

config = {
    "llm": {
        "provider": "openai",          # or "anthropic", "azure_openai", etc.
        "config": {
            "model": "gpt-4o-mini",
            "temperature": 0.1,        # low temp for deterministic extraction
        },
    },
    "embedder": {
        "provider": "openai",
        "config": {
            "model": "text-embedding-3-small",
            "embedding_dims": 1536,
        },
    },
    "vector_store": {
        "provider": "qdrant",          # or "chroma", "pgvector", "milvus"
        "config": {
            "host": "localhost",
            "port": 6333,
            "collection_name": "my_app_memories",
        },
    },
}

m = Memory.from_config(config)
```

**支持的供应商：**
- **LLM**：OpenAI、Anthropic、Azure OpenAI、Google AI、Groq、Ollama 等
- **嵌入模型**：OpenAI、Hugging Face、Google、Ollama
- **向量库**：Qdrant、Chroma、pgvector、Milvus、Pinecone、Weaviate

完整参考见 [Mem0 配置文档](https://docs.mem0.ai/open-source/configuration?utm_source=nirdiamant&utm_medium=github&utm_campaign=agent_memory_techniques)。

### 多用户隔离

每个 `user_id` 拥有自己隔离的记忆空间。
Carol 的花生过敏不会混进 Dave 的结果，尽管他们共用同一个 Mem0 实例。


In [ ]:
# ── 多用户隔离演示 ──
# 记忆严格按 user_id 隔离（不会串到别人）。

# 为两个不同用户各加一条记忆
m.add("I'm allergic to peanuts and I love spicy food", user_id="carol")
m.add("I adore peanut butter and I can't handle any spice", user_id="dave")

# 搜索饮食偏好，每个用户只能看到自己的
print("=== Carol's food preferences ===")
carol_food = m.search("food preferences and allergies", user_id="carol")
for mem in (carol_food.get("results", carol_food) if isinstance(carol_food, dict) else carol_food):
    if isinstance(mem, dict):
        print(f"  \u2022 {mem.get('memory', mem.get('text', str(mem)))}")

print()
print("=== Dave's food preferences ===")
dave_food = m.search("food preferences and allergies", user_id="dave")
for mem in (dave_food.get("results", dave_food) if isinstance(dave_food, dict) else dave_food):
    if isinstance(mem, dict):
        print(f"  \u2022 {mem.get('memory', mem.get('text', str(mem)))}")

print("\n\u2713 User isolation confirmed. No cross-contamination")

### 用元数据做分类

添加记忆时可以附加元数据（额外的标签）。
这样就能把记忆归入「travel（旅行）」「budget（预算）」等类别，便于结构化检索。


In [ ]:
# ── 用元数据做分类 ──
# 给记忆附加元数据，方便分类检索。

m.add(
    "I prefer window seats on flights and I'm in the Delta SkyMiles program",
    user_id="alice",
    metadata={"category": "travel_preferences"},
)

m.add(
    "My budget for dinner is usually around $30-40 per person",
    user_id="alice",
    metadata={"category": "budget"},
)

# 结合元数据查看记忆
print("=== All Alice's memories (with metadata) ===")
all_alice = m.get_all(user_id="alice")
mem_list = all_alice.get("results", all_alice) if isinstance(all_alice, dict) else all_alice
for mem in mem_list:
    text = mem.get("memory", mem.get("text", str(mem)))
    meta = mem.get("metadata", {})
    cat = meta.get("category", "auto") if meta else "auto"
    print(f"  [{cat}] {text}")

### 清理

删除所有演示记忆，保持本地 Qdrant 存储整洁。


In [ ]:
# ── 清理：删除演示记忆 ──
# 删除所有演示用户的记忆，保持存储整洁。

for uid in ["alice", "bob", "carol", "dave"]:
    m.delete_all(user_id=uid)
    print(f"\u2713 Deleted all memories for '{uid}'")

print("\n\u2713 Cleanup complete")

## 讨论与权衡

### 优势

- **接入代码极少：** 一次 `add()` 调用就完成抽取、去重和存储。无需构建自定义 NLP 流程、管理嵌入模型或手写向量库查询。
- **自动冲突解决：** 当用户的偏好或事实发生变化时，Mem0 能检测到矛盾并更新已存记忆。用 DIY 方式想把这个做对，难度超乎想象。
- **默认按用户隔离：** 按 `user_id` 划分记忆范围是内置能力。手写记忆系统最怕的跨用户串数据问题，这里天然避免。
- **自我改进：** 反复调用 `add()` 会打磨并合并相关记忆。记忆画像随时间变得更准确、更精炼。
- **供应商灵活：** 通过配置即可更换 LLM、嵌入模型和向量库，不用改应用代码。

### 劣势

- **抽取过程不透明：** 你无法精确控制抽取哪些内容。「什么值得记」由 Mem0 的 LLM 流程决定，未必符合你的业务需求。关键事实可能被漏掉，无关细节也可能被存进来。
- **每次添加都有 LLM 成本：** 每次 `add()` 都触发一次 LLM 抽取调用。高吞吐场景下，这笔开销相当可观。可以靠批处理缓解，但库本身不内置。
- **查询控制有限：** 语义搜索能力不错，但结构化查询就不那么方便了。比如「category = 'dietary' 且创建于 2025-01-01 之后的所有记忆」，用关系型数据库会容易得多。
- **本地 Qdrant 的局限：** 默认的本地 Qdrant 基于文件且单进程。生产部署需要正经的 Qdrant 服务器、pgvector 或云端向量库。
- **没有内置 TTL 或衰减：** TTL（Time To Live）指数据到期自动删除的规则。Mem0 的记忆会永久保留，除非你手动删除。长期运行的智能体需要在 Mem0 之上自己实现遗忘策略。

### 何时用 Mem0，何时自建

| 场景 | 建议 |
|----------|---------------|
| 原型化一个个性化聊天机器人 | **用 Mem0。** 拿到可用记忆的最快路径。 |
| 你需要精细控制存什么 | **自己建。** Mem0 的抽取不透明。 |
| 多用户应用，需要用户隔离 | **用 Mem0。** 隔离是内置的。 |
| 高吞吐（每分钟超过 1000 条消息） | **自己建。** 避免每条消息的 LLM 成本。 |
| 你需要记忆衰减、TTL 或遗忘机制 | **自己建**（或在 Mem0 之上叠加）。 |
| 团队想绕开记忆基础设施快速上线 | **用 Mem0。** 单一依赖，无需配置向量库。 |
| 你需要基于图的记忆（实体之间的关系） | **考虑 Graphiti 或 Zep。** Mem0 是扁平存储，不是图结构。 |

### Mem0 与其他记忆框架对比

| 核心机制 | Mem0 | Zep | Letta (MemGPT) |
|--------|------|-----|-----------------|
| 主要方式 | 扁平记忆库 | 时序知识图谱 | 自编辑记忆块 |
| 抽取方式 | 通过 LLM 自动抽取 | 对话分类 + 实体抽取 | 内心独白 |
| 冲突解决 | 内置 | 基于图 | 由智能体驱动 |
| 自托管 | 支持（开源版） | 支持（开源版） | 支持（开源版） |
| 托管云服务 | 有 | 有 | 有 |
| 最适合 | 简单直接的个性化 | 复杂关系 | 智能体式记忆管理 |


## 延伸阅读

- [Mem0 文档](https://docs.mem0.ai/?utm_source=nirdiamant&utm_medium=github&utm_campaign=agent_memory_techniques)：官方文档，覆盖所有操作、配置和集成。
- [Mem0 GitHub 仓库](https://github.com/mem0ai/mem0)：开源代码、issue 和示例。
- [PyPI 上的 Mem0 Python SDK](https://pypi.org/project/mem0ai/?utm_source=nirdiamant&utm_medium=github&utm_campaign=agent_memory_techniques)：包详情和版本历史。
- [Mem0 平台](https://app.mem0.ai/?utm_source=nirdiamant&utm_medium=github&utm_campaign=agent_memory_techniques)：带控制台和分析功能的托管云服务。
- [Anthropic：Building Effective Agents（2025）](https://www.anthropic.com/engineering/building-effective-agents?utm_source=nirdiamant&utm_medium=github&utm_campaign=agent_memory_techniques)：与记忆互补的智能体架构模式。
- [Letta (MemGPT)：自编辑记忆](https://github.com/letta-ai/letta)：另一种思路，智能体用内心独白管理记忆。
- [Zep：时序知识图谱](https://github.com/getzep/zep)：另一种思路，用带时间感知的图记忆。

---

*← 上一篇：[24 - 用 Graphiti 做图记忆](../24_graph_memory_graphiti/) · 下一篇：[26 - Letta (MemGPT) 模式](../26_letta_memgpt_patterns/) →*


## 🧪 动手试一试

三个小挑战，帮你加深理解。每个大约需要 10-30 分钟。

### 挑战 1：按用户隔离
用不同的 `user_id` 调用 `memory.add()`，创建两个用户。给每个用户存 5 条事实。然后针对每个用户分别调用 `memory.search()`，验证没有跨用户泄漏。把两组结果并排打印出来。

### 挑战 2：去重准确率
用 `memory.add()` 添加 10 条语义相似的事实（同一信息的不同表述）。调用 `memory.get_all()`，数一数 Mem0 实际保留了多少条独立记录、合并了多少条。算出去重率，并记下它保留重复项或错误合并不同事实的情况。

### 挑战 3：完整的记忆生命周期
让一条事实走完整个生命周期：先 `add()`，再 `search()` 找到它，用新信息 `update()`，再 `search()` 确认更新生效，最后 `delete()` 删除。验证它不再出现在 `get_all()` 里。这条端到端流程对应技术 23「工具化记忆」中基于工具的控制模式。


![](https://europe-west1-amt-views-tracker.cloudfunctions.net/amt-tracker?notebook=all-techniques--25-mem0-patterns--mem0-patterns)
